In [ ]:
# Step 1: Install requirements and mount Drive
!pip install ultralytics -q
import os

from google.colab import drive
drive.mount('/content/drive')

# Define your specific Drive paths
drive_zip_path = '/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip'
local_extract_path = '/content/dataset'

# Extract the dataset to fast local storage
if not os.path.exists(local_extract_path):
    print("📦 Extracting dataset from Drive... This might take a minute.")
    os.makedirs(local_extract_path, exist_ok=True)
    !unzip -q "{drive_zip_path}" -d "{local_extract_path}"
    print("✅ Dataset extracted successfully to local Colab storage!")
else:
    print("✅ Dataset already extracted.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.8 MB/s eta 0:00:00
Mounted at /content/drive
📦 Extracting dataset from Drive... This might take a minute.
/content/dataset/images/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.jpg  bad CRC cd8705f0  (should be 334d85a6)
✅ Dataset extracted successfully to local Colab storage!


In [ ]:
import os

# 1. Delete from the currently extracted Colab folder
bad_img_local = '/content/dataset/images/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.jpg'
bad_lbl_local = '/content/dataset/labels/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.txt'

if os.path.exists(bad_img_local):
    os.remove(bad_img_local)
    print("🧹 Removed corrupted image from local storage.")
if os.path.exists(bad_lbl_local):
    os.remove(bad_lbl_local)
    print("🧹 Removed corresponding label from local storage.")

# 2. PERMANENTLY delete from the ZIP file in your Google Drive
drive_zip = '/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip'

# These are the relative paths as they exist inside the zip archive
bad_img_zip = 'images/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.jpg'
bad_lbl_zip = 'labels/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.txt'

print("🔥 Executing surgical deletion on the master Google Drive ZIP...")
!zip -d "{drive_zip}" "{bad_img_zip}" "{bad_lbl_zip}"
print("✅ Corrupted files have been permanently eradicated from your dataset!")

🧹 Removed corrupted image from local storage.
🧹 Removed corresponding label from local storage.
🔥 Executing surgical deletion on the master Google Drive ZIP...
deleting: images/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.jpg
deleting: labels/val/frame526_jpg.rf.815d398b6f5479b3200c6e2956703c1e.txt
✅ Corrupted files have been permanently eradicated from your dataset!


In [ ]:
import os
import yaml
import torch
import torch.nn as nn

# Define your main Drive folder
drive_dir = '/content/drive/MyDrive/Thesis/version5_2'
os.makedirs(drive_dir, exist_ok=True)

# ==========================================
# 1. CREATE DATA YAML (Saved to Drive)
# ==========================================
data_config = {
    'path': '/content/dataset',  # The images are in fast local storage
    'train': 'images/train',
    'val': 'images/val',
    'nc': 9,
    'names': ['car', 'truck', 'bus', 'motorcycle', 'bicycle', 'pedestrian', 'auto_rickshaw', 'rickshaw', 'van']
}
data_yaml_path = os.path.join(drive_dir, 'fixed_data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)
print(f"✅ fixed_data.yaml permanently saved to: {data_yaml_path}")

# ==========================================
# 2. DEFINE & INJECT DCFM_v2
# ==========================================
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2
print("✅ DCFM_v2 Module injected into Ultralytics core.")

# ==========================================
# 3. EXACT 1:1 CBAM-MATCHED ARCHITECTURE (Saved to Drive)
# ==========================================
arch_yaml = """
nc: 9
scales:
  # The true Medium scale matched from your CBAM file: [depth, width, max_channels]
  m: [0.50, 1.00, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  # === NOVEL DCFM MODULE (Replaces CBAM at the exact same location) ===
  - [-1, 1, DCFM_v2, [512]]

  - [[16, 19, 23], 1, Detect, [nc]]
"""
arch_yaml_path = os.path.join(drive_dir, 'yolov11m-dcfm.yaml')
with open(arch_yaml_path, 'w') as f:
    f.write(arch_yaml)
print(f"✅ yolov11m-dcfm.yaml permanently saved to: {arch_yaml_path}")

✅ fixed_data.yaml permanently saved to: /content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ DCFM_v2 Module injected into Ultralytics core.
✅ yolov11m-dcfm.yaml permanently saved to: /content/drive/MyDrive/Thesis/version5_2/yolov11m-dcfm.yaml


In [ ]:
from ultralytics import YOLO
import torch

# Clear any background memory before starting
torch.cuda.empty_cache()

print("🚀 LAUNCHING THE FINAL DCFM THESIS RUN ON COLAB T4!")

# 1. Load your custom architecture from Drive and transfer base weights
model_yaml = '/content/drive/MyDrive/Thesis/version5_2/yolov11m-dcfm.yaml'
model = YOLO(model_yaml).load('yolo11m.pt')

# 2. Start the 50-Epoch Master Training
# Everything saves directly to your Google Drive so nothing is ever lost
results = model.train(
    data='/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml',
    epochs=50,
    imgsz=640,            # 🎯 Full resolution target!
    batch=4,              # 🛡️ Reduced to 4 to safely fit inside the 15GB T4 limit
    nbs=64,               # 🛡️ Accumulates gradients to simulate a large batch size for accuracy
    project='/content/drive/MyDrive/Thesis/version5_2/thesis_results',
    name='dcfm_master_run',
    plots=True,
    save_period=1,        # Saves a checkpoint every single epoch to Drive
    verbose=True,
    amp=True              # Ensures Mixed Precision is on to save VRAM
)

print("✅ IEEE DCFM ABLATION STUDY COMPLETE!")

🚀 LAUNCHING THE FINAL DCFM THESIS RUN ON COLAB T4!
Transferred 528/659 items from pretrained weights
Ultralytics 8.4.28 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Thesis/version5_2/yolov11m

In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.7 MB/s eta 0:00:00


In [ ]:
# 1. Mount Drive & Extract Dataset (Colab resets temporary storage overnight)
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"

# 2. Install YOLO
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

# 3. Re-Inject DCFM_v2 (Crucial: PyTorch needs to remember your custom math)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM LAST SAVED EPOCH...")

# 4. Resume Command
# We point YOLO directly to the 'last.pt' file sitting in your Drive
model = YOLO('/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt')

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state!
results = model.train(resume=True)

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM LAST SAVED EPOCH...
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=tor

In [ ]:
# 1. Mount Drive & Extract Dataset
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/dataset/images/train'):
    print("📦 Extracting dataset from Drive to fast local storage...")
    !unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"
    print("✅ Dataset ready!")

# 2. Install Ultralytics
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

# 3. Re-Inject DCFM_v2 (Crucial: PyTorch needs this to read your custom weights)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM LAST SAVED EPOCH...")

# 4. Resume Command
# We point YOLO directly to the 'last.pt' file sitting in your Drive
last_weight_path = '/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt'
model = YOLO(last_weight_path)

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state!
results = model.train(resume=True)

Mounted at /content/drive
📦 Extracting dataset from Drive to fast local storage...
✅ Dataset ready!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM LAST SAVED EPOCH...
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, 

In [ ]:
# 1. Mount Drive & Extract Dataset
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/dataset/images/train'):
    print("📦 Extracting dataset from Drive to fast local storage...")
    !unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"
    print("✅ Dataset ready!")

# 2. Install Ultralytics & Clear Cache
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

torch.cuda.empty_cache()

# 3. Re-Inject DCFM_v2 (Crucial for PyTorch to read your custom weights)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM EPOCH 28...")

# 4. Resume Command
# Pointing YOLO directly to the 'last.pt' file sitting in your Drive
last_weight_path = '/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt'
model = YOLO(last_weight_path)

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state
results = model.train(resume=True)

Mounted at /content/drive
📦 Extracting dataset from Drive to fast local storage...
✅ Dataset ready!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM EPOCH 28...
Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

In [ ]:
# 1. Mount Drive & Extract Dataset
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/dataset/images/train'):
    print("📦 Extracting dataset from Drive to fast local storage...")
    !unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"
    print("✅ Dataset ready!")

# 2. Install Ultralytics & Clear Cache
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

torch.cuda.empty_cache()

# 3. Re-Inject DCFM_v2 (Crucial for PyTorch to read your custom weights)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM EPOCH 28...")

# 4. Resume Command
# Pointing YOLO directly to the 'last.pt' file sitting in your Drive
last_weight_path = '/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt'
model = YOLO(last_weight_path)

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state
results = model.train(resume=True)

Mounted at /content/drive
📦 Extracting dataset from Drive to fast local storage...
✅ Dataset ready!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM EPOCH 28...
Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

In [ ]:
# 1. Mount Drive & Extract Dataset
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/dataset/images/train'):
    print("📦 Extracting dataset from Drive to fast local storage...")
    !unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"
    print("✅ Dataset ready!")

# 2. Install Ultralytics & Clear Cache
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

torch.cuda.empty_cache()

# 3. Re-Inject DCFM_v2 (Crucial for PyTorch to read your custom weights)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM EPOCH 28...")

# 4. Resume Command
# Pointing YOLO directly to the 'last.pt' file sitting in your Drive
last_weight_path = '/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt'
model = YOLO(last_weight_path)

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state
results = model.train(resume=True)

Mounted at /content/drive
📦 Extracting dataset from Drive to fast local storage...
✅ Dataset ready!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM EPOCH 28...
Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

In [ ]:
# 1. Mount Drive & Extract Dataset
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/dataset/images/train'):
    print("📦 Extracting dataset from Drive to fast local storage...")
    !unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"
    print("✅ Dataset ready!")

# 2. Install Ultralytics & Clear Cache
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

torch.cuda.empty_cache()

# 3. Re-Inject DCFM_v2 (Crucial for PyTorch to read your custom weights)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM EPOCH 28...")

# 4. Resume Command
# Pointing YOLO directly to the 'last.pt' file sitting in your Drive
last_weight_path = '/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt'
model = YOLO(last_weight_path)

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state
results = model.train(resume=True)

Mounted at /content/drive
📦 Extracting dataset from Drive to fast local storage...
✅ Dataset ready!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM EPOCH 28...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

In [ ]:
# 1. Mount Drive & Extract Dataset
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/dataset/images/train'):
    print("📦 Extracting dataset from Drive to fast local storage...")
    !unzip -q "/content/drive/MyDrive/Thesis/version5_2/v_5 dataset.zip" -d "/content/dataset"
    print("✅ Dataset ready!")

# 2. Install Ultralytics & Clear Cache
!pip install ultralytics -q
import torch
import torch.nn as nn
from ultralytics import YOLO

torch.cuda.empty_cache()

# 3. Re-Inject DCFM_v2 (Crucial for PyTorch to read your custom weights)
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 512
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)

    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2

print("🚀 RESUMING TRAINING FROM EPOCH 28...")

# 4. Resume Command
# Pointing YOLO directly to the 'last.pt' file sitting in your Drive
last_weight_path = '/content/drive/MyDrive/Thesis/version5_2/thesis_results/dcfm_master_run/weights/last.pt'
model = YOLO(last_weight_path)

# resume=True tells YOLO to automatically pick up the epoch count and optimizer state
results = model.train(resume=True)

Mounted at /content/drive
📦 Extracting dataset from Drive to fast local storage...
✅ Dataset ready!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 RESUMING TRAINING FROM EPOCH 28...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/version5_2/fixed_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.4 MB/s eta 0:00:00
